In [1]:
## Gerekli paketleri kuruyoruz (bu defteri ilk kez açan biri sadece bu cell'i
## çalıştırıp ardından diğer cell'leri sırayla çalıştırarak kullanabilir).
!uv pip install -q pandas pyarrow sentence-transformers torch transformers peft huggingface_hub openai ollama

In [2]:
import json
import os
import time
import getpass

import pandas as pd
from sentence_transformers import SentenceTransformer

# Anlamsal benzerlik kontrolü için modeli başlatıyoruz
anlamsal_benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


# Verilen cevabın doğru olup olmadığını kontrol eden fonksiyon
def cevap_dogru_mu(dogru_cevap_index, verilen_cevap, secenekler):
    # Seçenekler A, B, C, D, E şeklinde dizildiği için harfler listesini tanımlıyoruz
    harfler = ['A', 'B', 'C', 'D', 'E']

    # Doğru cevaba karşılık gelen harfi belirliyoruz
    dogru_harf = harfler[dogru_cevap_index]

    # Kullanıcının verdiği cevabı büyük harflere çeviriyoruz, çünkü karşılaştırmada harf duyarlılığı istemiyoruz
    verilen_cevap = verilen_cevap.upper()

    # Verilen cevabın başındaki ve sonundaki boşlukları temizliyoruz
    verilen_cevap = verilen_cevap.strip()

    # Eğer verilen cevap doğrudan doğru harfe eşitse doğru kabul ediyoruz
    if dogru_harf == verilen_cevap:
        return True
    # Eğer cevap birden fazla karakter içeriyor ve 2. karakter boşluk, noktalama gibi özel karakterse,
    # sadece ilk harfi kontrol ediyoruz
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", "."]:
        return dogru_harf == verilen_cevap[0]
    else:
        # Anlamsal benzerlik modelini kullanarak verilen cevap ve seçenekleri vektörel olarak kodluyoruz
        encoded_cevap = anlamsal_benzerlik_modeli.encode([verilen_cevap])
        encoded_secenekler = anlamsal_benzerlik_modeli.encode(secenekler)

        # Kodlanan cevap ile seçenekler arasındaki benzerlik puanlarını hesaplıyoruz
        benzerlik_listesi = anlamsal_benzerlik_modeli.similarity(encoded_cevap, encoded_secenekler).tolist()[0]

        # Benzerlik puanlarının en yükseğini buluyoruz
        en_yuksek_benzerlik = max(benzerlik_listesi)

        # En yüksek benzerlik puanının hangi seçeneğe ait olduğunu buluyoruz
        en_yuksek_benzerlik_index = benzerlik_listesi.index(en_yuksek_benzerlik)

        # Eğer en yüksek benzerlik doğru cevabın indeksine eşitse, doğru cevabı bulmuşuz demektir
        return en_yuksek_benzerlik_index == dogru_cevap_index


# İlerleme çubuğu fonksiyonu
def ilerleme_cubugu(guncel, toplam, cubuk_uzunlugu=40):
    ilerleme = guncel / toplam
    blok = int(cubuk_uzunlugu * ilerleme)
    cubuk = "#" * blok + "-" * (cubuk_uzunlugu - blok)
    return f"[{cubuk}] {ilerleme * 100:.2f}%"


# ---------------------------------------------------------------------------
# Backend'ler: MMLU test döngüsü, model çağrısını bu ortak arayüz üzerinden yapar.
# Her backend "cevap_uret(prompt) -> str" ve "model_detay() -> dict" sağlar,
# ayrıca sonuçları eşsiz şekilde etiketlemek için "model_adi" alanı bulunur.
# ---------------------------------------------------------------------------

class HuggingFaceBackend:
    """HuggingFace Hub'dan fine-tune modelini (LoRA adaptörü + taban model) indirip
    lokalde (MPS/CUDA/CPU) çalıştırır. Adaptör repository'si otomatik tespit edilip
    taban modelin üzerine yüklenir."""

    def __init__(self):
        import torch
        from huggingface_hub import hf_hub_download
        from huggingface_hub.errors import EntryNotFoundError
        from transformers import AutoModelForCausalLM, AutoTokenizer

        model_repo = input(
            "HuggingFace model repo id [SalihHub/trendyol-marangoz-finetuned-gemma-4-E4B-it]: "
        ).strip() or "SalihHub/trendyol-marangoz-finetuned-gemma-4-E4B-it"

        adapter_config = None
        try:
            adapter_config_yolu = hf_hub_download(model_repo, "adapter_config.json")
            with open(adapter_config_yolu) as f:
                adapter_config = json.load(f)
        except EntryNotFoundError:
            adapter_config = None

        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
        self.device = device
        self.dtype = torch.bfloat16 if device != "cpu" else torch.float32

        if adapter_config is not None:
            # Bu repo bir LoRA adaptörü: taban modeli indirip adaptörü üzerine bindiriyoruz.
            from peft import LoraConfig, PeftModel

            onerilen_taban = adapter_config.get('base_model_name_or_path', '')
            if "unsloth" in onerilen_taban and "bnb-4bit" in onerilen_taban:
                # Unsloth'un 4bit (bnb) taban modelleri CUDA gerektirir; Mac/CPU/MPS için
                # aynı mimarinin quantize edilmemiş halini kullanıyoruz (LoRA farkı, taban
                # modelin quantization seviyesinden bağımsız olarak uygulanabilir).
                onerilen_taban = "google/gemma-4-E4B-it"

            taban_model_repo = input(
                f"Taban (base) model repo id [{onerilen_taban}]: "
            ).strip() or onerilen_taban

            print(f"Taban model indiriliyor: {taban_model_repo} (bu işlem uzun sürebilir)")
            base_model = AutoModelForCausalLM.from_pretrained(
                taban_model_repo, dtype=self.dtype, low_cpu_mem_usage=True
            )

            print(f"LoRA adaptörü yükleniyor: {model_repo}")
            # Bu adaptör; dil modeli decoder katmanlarının yanı sıra vision/audio kule
            # katmanlarını da (embed_vision/embed_audio projeksiyonları, vision_tower
            # patch_embedder) hedefliyor. O katmanlardaki bazı projeksiyonlar transformers'ın
            # Gemma4ClippableLinear sarmalayıcısını kullanıyor ve peft 0.19 bu sınıfı
            # desteklemiyor ("Target module ... is not supported" hatası). MMLU testi salt
            # metin tabanlı olduğu ve vision/audio kuleleri metin üretiminde hiç
            # kullanılmadığı için LoRA'yı yalnızca gerçek nn.Linear olan projeksiyonlara
            # uyguluyoruz. Bu katmanların tam yolu (örn. "model.layers..." veya
            # "model.language_model.layers...") mimariye göre değiştiğinden regex ile
            # tahmin etmek yerine base modeli introspect ederek buluyoruz.
            proj_ekleri = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
            lora_hedef_modulleri = [
                isim for isim, modul in base_model.named_modules()
                if isim.split('.')[-1] in proj_ekleri and isinstance(modul, torch.nn.Linear)
            ]
            if not lora_hedef_modulleri:
                raise RuntimeError(
                    "Taban modelde LoRA için uygun (nn.Linear) projeksiyon modülü bulunamadı."
                )

            lora_config = LoraConfig.from_pretrained(model_repo)
            lora_config.target_modules = lora_hedef_modulleri
            self.model = PeftModel.from_pretrained(base_model, model_repo, config=lora_config)
            self.taban_model_repo = taban_model_repo
        else:
            print(f"Model indiriliyor: {model_repo} (bu işlem uzun sürebilir)")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_repo, dtype=self.dtype, low_cpu_mem_usage=True
            )
            self.taban_model_repo = model_repo

        self.tokenizer = AutoTokenizer.from_pretrained(model_repo)
        self.model.to(device)
        self.model.eval()

        self.model_repo = model_repo
        self.model_adi = "hf_" + model_repo.split("/")[-1]

    def cevap_uret(self, prompt):
        import torch

        mesajlar = [{"role": "user", "content": prompt}]
        # Gemma4 çok modlu (vision/audio/text) bir model olduğu için tokenizer'ın
        # apply_chat_template'i düz bir tensor değil, input_ids + attention_mask
        # içeren bir BatchEncoding (dict) döndürüyor. return_dict=True ile bunu
        # açıkça isteyip **girdi şeklinde generate'e veriyoruz.
        girdi = self.tokenizer.apply_chat_template(
            mesajlar, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(self.device)

        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = self.tokenizer.eos_token_id

        with torch.no_grad():
            cikti = self.model.generate(
                **girdi,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=pad_token_id,
            )

        girdi_uzunlugu = girdi["input_ids"].shape[-1]
        yeni_tokenlar = cikti[0][girdi_uzunlugu:]
        return self.tokenizer.decode(yeni_tokenlar, skip_special_tokens=True)

    def model_detay(self):
        toplam_parametre = sum(p.numel() for p in self.model.parameters())
        return {
            'model': self.model_adi,
            'format': 'safetensors',
            'family': getattr(self.model.config, 'model_type', 'bilinmiyor'),
            'parameter_size': f"{toplam_parametre / 1e9:.2f}B",
            'quantization_level': str(self.dtype).replace('torch.', ''),
        }


class OpenRouterBackend:
    """OpenRouter üzerinden, OpenAI uyumlu API ile istediğin herhangi bir modeli çalıştırır."""

    def __init__(self):
        from openai import OpenAI

        api_key = os.environ.get("OPENROUTER_API_KEY")
        if not api_key:
            api_key = getpass.getpass("OpenRouter API anahtarı (OPENROUTER_API_KEY): ").strip()

        self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

        model_ismi = ''
        while not model_ismi:
            model_ismi = input("OpenRouter model ismi (örn. google/gemma-4-e4b-it): ").strip()

        self.model_ismi = model_ismi
        self.model_adi = "openrouter_" + model_ismi.replace("/", "_")

    def cevap_uret(self, prompt):
        yanit = self.client.chat.completions.create(
            model=self.model_ismi,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=16,
            temperature=0,
        )
        return yanit.choices[0].message.content or ""

    def model_detay(self):
        return {
            'model': self.model_adi,
            'format': 'api',
            'family': 'openrouter',
            'parameter_size': 'bilinmiyor',
            'quantization_level': 'bilinmiyor',
        }


class OllamaBackend:
    """Lokalde `ollama pull`/`ollama create` ile kurulmuş modelleri çalıştırır
    (LoRA'yı merge edip GGUF olarak Ollama'ya import ettiysen buradan da test edebilirsin)."""

    def __init__(self):
        import ollama
        self._ollama = ollama

        mevcut_modeller = [model['model'] for model in ollama.list()['models']]
        for model in mevcut_modeller:
            print(model)

        secilen_model_ismi = ''
        while secilen_model_ismi not in mevcut_modeller:
            secilen_model_ismi = input('Model ismi: ')

        self.model_ismi = secilen_model_ismi
        self.model_adi = secilen_model_ismi
        self._model_bilgi = ollama.list()['models'][mevcut_modeller.index(secilen_model_ismi)]

    def cevap_uret(self, prompt):
        response = self._ollama.generate(
            model=self.model_ismi,
            prompt=prompt,
            think=False,
            options=self._ollama.Options(seed=42, num_predict=42),
        )
        return response['response']

    def model_detay(self):
        detay = self._model_bilgi['details']
        return {
            'model': self.model_adi,
            'format': detay['format'],
            'family': detay['family'],
            'parameter_size': detay['parameter_size'],
            'quantization_level': detay['quantization_level'],
        }


# Verilerimizi okuyoruz (yalnızca sorular; sonuçları Hub'a göndermiyoruz, yerelde JSON olarak tutuyoruz)
mmlu_veri = pd.read_parquet("hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet")


# Model test fonksiyonu. Sonuçları {model_adi}_mmlu_sonuc.json dosyasına yazar ve aynı
# json'u (model adı, özet istatistikler, bölüm kırılımı, soru bazlı işaretlemeler) döner.
def modeli_test_et(backend):
    model_detayli_sonuc = backend.model_detay()
    model_bolum_sonuc = {}

    cevap_kolonu = backend.model_adi + '_cevap'
    dogru_mu_kolonu = backend.model_adi + '_dogru_mu'

    baslama_zamani = time.time()
    dogru_cevap_sayisi = 0

    # Soruları test etme ve cevap kontrolü
    for i in range(len(mmlu_veri)):
        soru = mmlu_veri.iloc[i]['soru']
        soru += "\n"
        harfler = ['A', 'B', 'C', 'D', 'E']
        for j in range(len(mmlu_veri.iloc[i]['secenekler'])):
            secenek = mmlu_veri.iloc[i]['secenekler'][j]
            soru += harfler[j] + ": " + secenek + "\n"

        prompt = "Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun doğru cevabı olduğunu yaz. Örneğin 'A' veya 'B' gibi. Lütfen herhangi bir açıklama yapma!\nSoru: " + soru

        cevap = backend.cevap_uret(prompt)

        bolum = mmlu_veri.iloc[i]['bolum']

        if bolum not in model_bolum_sonuc:
            model_bolum_sonuc[bolum] = 0

        sonuc = cevap_dogru_mu(mmlu_veri.iloc[i]['cevap'], cevap, mmlu_veri.iloc[i]['secenekler'])

        # Cevabı ve doğru/yanlış işaretini veriye ekleme
        mmlu_veri.loc[i, cevap_kolonu] = cevap
        mmlu_veri.loc[i, dogru_mu_kolonu] = sonuc

        if sonuc:
            dogru_cevap_sayisi += 1
            model_bolum_sonuc[bolum] += 1

        soru_index = i + 1
        simdi = time.time()
        cubuk = ilerleme_cubugu(soru_index, len(mmlu_veri))

        print(f"\r{soru_index} soru çözüldü. Geçen süre: {round(simdi - baslama_zamani, 3)} saniye. Doğru cevap sayısı: {dogru_cevap_sayisi}. Başarı: {round(dogru_cevap_sayisi / soru_index, 4)} İlerleme: {cubuk}", end="")

    ortalama = round(dogru_cevap_sayisi / len(mmlu_veri), 4)
    bitis_zamani = time.time()

    model_detayli_sonuc['dogru_cevap_sayisi'] = dogru_cevap_sayisi
    model_detayli_sonuc['basari'] = ortalama * 100
    model_detayli_sonuc['toplam_sure'] = round(bitis_zamani - baslama_zamani, 3)

    print(f"\n{soru_index} soru çözüldü. Geçen süre: {round(bitis_zamani - baslama_zamani, 3)} saniye. Doğru cevap sayısı: {dogru_cevap_sayisi}")

    # Soru bazlı işaretlemeler: modelin cevabı ve doğru/yanlış durumu
    isaretlemeler = []
    for i in range(len(mmlu_veri)):
        satir = mmlu_veri.iloc[i]
        isaretlemeler.append({
            'soru': satir['soru'],
            'bolum': satir['bolum'],
            'dogru_cevap_index': int(satir['cevap']),
            'secenekler': list(satir['secenekler']),
            'model_cevabi': satir[cevap_kolonu],
            'dogru_mu': bool(satir[dogru_mu_kolonu]),
        })

    json_sonuc = {
        'model': backend.model_adi,
        'ozet': model_detayli_sonuc,
        'bolum_sonuclari': model_bolum_sonuc,
        'isaretlemeler': isaretlemeler,
    }

    dosya_adi = f"{backend.model_adi}_mmlu_sonuc.json"
    with open(dosya_adi, 'w', encoding='utf-8') as f:
        json.dump(json_sonuc, f, ensure_ascii=False, indent=2)
    print(f"Sonuçlar '{dosya_adi}' dosyasına kaydedildi.")

    return json_sonuc

/Users/salih/Desktop/magibu-work/magibu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11519.63it/s]


In [3]:
## HuggingFace (taban model + LoRA merge) ile fine tuned model testi
## Not: Taban modeli ayrıca Ollama'ya indirsen bile MMLU testi burada
## transformers+peft ile taban model üzerine LoRA adaptörünü merge ederek çalışır.
backend_hf = HuggingFaceBackend()
sonuc_hf = modeli_test_et(backend_hf)
sonuc_hf

Taban model indiriliyor: google/gemma-4-E4B-it (bu işlem uzun sürebilir)


Loading weights: 100%|██████████| 2076/2076 [00:00<00:00, 14546.57it/s]


LoRA adaptörü yükleniyor: SalihHub/trendyol-marangoz-finetuned-gemma-4-E4B-it
6200 soru çözüldü. Geçen süre: 2273.375 saniye. Doğru cevap sayısı: 4447. Başarı: 0.7173 İlerleme: [########################################] 100.00%
6200 soru çözüldü. Geçen süre: 2273.375 saniye. Doğru cevap sayısı: 4447
Sonuçlar 'hf_trendyol-marangoz-finetuned-gemma-4-E4B-it_mmlu_sonuc.json' dosyasına kaydedildi.


{'model': 'hf_trendyol-marangoz-finetuned-gemma-4-E4B-it',
 'ozet': {'model': 'hf_trendyol-marangoz-finetuned-gemma-4-E4B-it',
  'format': 'safetensors',
  'family': 'gemma4',
  'parameter_size': '7.96B',
  'quantization_level': 'bfloat16',
  'dogru_cevap_sayisi': 4447,
  'basari': 71.73,
  'toplam_sure': 2273.375},
 'bolum_sonuclari': {'AUZEF': 70,
  'Acil Durum ve Afet Yönetimi': 78,
  'Adalet': 73,
  'Aşçılık': 68,
  'Bankacılık ve Sigortacılık': 75,
  'Büro Yönetimi ve Yönetici Asistanlığı': 73,
  'DHBT': 61,
  'Dini Bilgiler': 80,
  'Dış Ticaret': 73,
  'Ehliyet Sınavı': 89,
  'Elektrik Enerjisi Üretim,İletim ve Dağıtımı': 73,
  'Emlak ve Emlak Yönetimi': 65,
  'Ev İdaresi': 75,
  'Felsefe': 81,
  'Fotoğrafçılık ve Kameramanlık': 74,
  'Futbol': 57,
  'Halkla İlişkiler ve Reklamcılık': 77,
  'Halkla İlişkiler ve Tanıtım': 75,
  'Havacılık Yönetimi': 72,
  'KPSS': 57,
  'KPSS Denemeleri': 61,
  'Kamu Yönetimi': 71,
  'Kim 500 Milyar İster': 56,
  'Kültürel Miras ve Turizm': 73,
  '

In [4]:
## OpenRouter ile farklı modelleri test etme
backend_openrouter = OpenRouterBackend()
sonuc_openrouter = modeli_test_et(backend_openrouter)
sonuc_openrouter

6200 soru çözüldü. Geçen süre: 5416.363 saniye. Doğru cevap sayısı: 5263. Başarı: 0.8489 İlerleme: [########################################] 100.00%
6200 soru çözüldü. Geçen süre: 5416.363 saniye. Doğru cevap sayısı: 5263
Sonuçlar 'openrouter_google_gemma-4-31b-it:nitro_mmlu_sonuc.json' dosyasına kaydedildi.


{'model': 'openrouter_google_gemma-4-31b-it:nitro',
 'ozet': {'model': 'openrouter_google_gemma-4-31b-it:nitro',
  'format': 'api',
  'family': 'openrouter',
  'parameter_size': 'bilinmiyor',
  'quantization_level': 'bilinmiyor',
  'dogru_cevap_sayisi': 5263,
  'basari': 84.89,
  'toplam_sure': 5416.363},
 'bolum_sonuclari': {'AUZEF': 88,
  'Acil Durum ve Afet Yönetimi': 90,
  'Adalet': 90,
  'Aşçılık': 87,
  'Bankacılık ve Sigortacılık': 88,
  'Büro Yönetimi ve Yönetici Asistanlığı': 83,
  'DHBT': 84,
  'Dini Bilgiler': 96,
  'Dış Ticaret': 86,
  'Ehliyet Sınavı': 95,
  'Elektrik Enerjisi Üretim,İletim ve Dağıtımı': 86,
  'Emlak ve Emlak Yönetimi': 87,
  'Ev İdaresi': 89,
  'Felsefe': 89,
  'Fotoğrafçılık ve Kameramanlık': 89,
  'Futbol': 79,
  'Halkla İlişkiler ve Reklamcılık': 86,
  'Halkla İlişkiler ve Tanıtım': 91,
  'Havacılık Yönetimi': 85,
  'KPSS': 69,
  'KPSS Denemeleri': 80,
  'Kamu Yönetimi': 84,
  'Kim 500 Milyar İster': 88,
  'Kültürel Miras ve Turizm': 81,
  'Laborant ve